# Double Barrier Options & SABR Smile

This notebook covers two complementary extensions:

1. **Double barrier options** — European options that are knocked out when the spot touches *either* a lower barrier $L$ or an upper barrier $U$.  Priced via eigenfunction expansion of the absorbed GBM transition density.
2. **SABR stochastic volatility** — The Hagan et al. (2002) lognormal approximation for European option prices under the SABR model `dF = α F^β dW₁, dα = ν α dW₂`.

**References**
- Kunitomo, N. & Ikeda, M. (1992). *Pricing Options with Curved Boundaries*. Mathematical Finance 2(4).
- Hagan, P. et al. (2002). *Managing Smile Risk*. Wilmott Magazine, September 2002.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from foureng.analytics.bsm_barrier import bsm_call, bsm_put, bsm_barrier_price, bsm_double_barrier_price
from foureng.models.sabr import SabrParams, sabr_hagan_implied_vol
from foureng.models.base import ForwardSpec
from foureng.pricers.sabr import sabr_hagan_price_at_strikes
from foureng.pipeline import price_strip

# Shared BSM parameters
S0 = 100.0
r, q, sigma, T = 0.05, 0.0, 0.20, 1.0

---
## Part 1 — Double Barrier Options

A **double knock-out** call/put pays the vanilla payoff at maturity $T$ *only if* the spot stays within the corridor $(L, U)$ throughout $[0, T]$.

The eigenfunction expansion of the absorbed GBM between $[\log L, \log U]$:

$$p(x,T;x_0) = \frac{2}{b-a}\sum_{n=1}^{\infty} e^{-\lambda_n T}\, e^{\mu(x-x_0)/\sigma^2 - \mu^2 T/(2\sigma^2)}\, \sin\!\left(\frac{n\pi(x_0-a)}{b-a}\right)\sin\!\left(\frac{n\pi(x-a)}{b-a}\right)$$

where $a=\ln L$, $b=\ln U$, $\mu = r-q-\sigma^2/2$, $\lambda_n = \frac{n^2\pi^2\sigma^2}{2(b-a)^2}$.

### 1.1  In-out parity

In [ ]:
K, L, U = 100.0, 80.0, 130.0

vanilla_call = bsm_call(S0, K, r, q, T, sigma)
vanilla_put  = bsm_put(S0, K, r, q, T, sigma)

dko_call = bsm_double_barrier_price(S0, K, L, U, r, q, T, sigma, cp=1, barrier_type="double_out")
dki_call = bsm_double_barrier_price(S0, K, L, U, r, q, T, sigma, cp=1, barrier_type="double_in")

dko_put = bsm_double_barrier_price(S0, K, L, U, r, q, T, sigma, cp=-1, barrier_type="double_out")
dki_put = bsm_double_barrier_price(S0, K, L, U, r, q, T, sigma, cp=-1, barrier_type="double_in")

print(f'Vanilla call:          {vanilla_call:.4f}')
print(f'Double KO call:        {dko_call:.4f}')
print(f'Double KI call:        {dki_call:.4f}')
print(f'KO + KI  (call):       {dko_call+dki_call:.4f}  (error={abs(dko_call+dki_call-vanilla_call):.2e})')
print()
print(f'Vanilla put:           {vanilla_put:.4f}')
print(f'Double KO put:         {dko_put:.4f}')
print(f'Double KI put:         {dki_put:.4f}')
print(f'KO + KI  (put):        {dko_put+dki_put:.4f}  (error={abs(dko_put+dki_put-vanilla_put):.2e})')

### 1.2  Barrier width vs price

In [ ]:
# Symmetric corridors: L = S0 / (1+δ), U = S0 * (1+δ)
deltas = np.linspace(0.05, 0.90, 50)
dko_prices, dki_prices, single_do_prices = [], [], []

for d in deltas:
    L_d = S0 / (1.0 + d)
    U_d = S0 * (1.0 + d)
    dko_prices.append(bsm_double_barrier_price(S0, K, L_d, U_d, r, q, T, sigma, cp=1))
    dki_prices.append(bsm_double_barrier_price(S0, K, L_d, U_d, r, q, T, sigma, cp=1, barrier_type="double_in"))
    single_do_prices.append(bsm_barrier_price(S0, K, L_d, r, q, T, sigma, 'down_out', cp=1))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(deltas, dko_prices, 'b-',  label='Double KO call')
ax.plot(deltas, dki_prices, 'r--', label='Double KI call')
ax.plot(deltas, single_do_prices, 'g:', lw=1.5, label='Single down-out call (lower barrier only)')
ax.axhline(vanilla_call, color='k', lw=1, ls='-.',  label=f'Vanilla call = {vanilla_call:.3f}')
ax.set_xlabel('Half-width δ  (L = S₀/(1+δ), U = S₀(1+δ))')
ax.set_ylabel('Call price')
ax.set_title('Double barrier call price vs corridor width')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### 1.3  Strike sweep — double barrier smile

In [ ]:
strikes = np.linspace(82, 128, 50)
L_fixed, U_fixed = 80.0, 130.0

dko_vs_K = [bsm_double_barrier_price(S0, k, L_fixed, U_fixed, r, q, T, sigma, cp=1) for k in strikes]
vanilla_vs_K = [bsm_call(S0, k, r, q, T, sigma) for k in strikes]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(strikes, vanilla_vs_K, 'k--', label='Vanilla BSM call', lw=1.5)
ax.plot(strikes, dko_vs_K, 'b-',  label=f'Double KO call  (L={L_fixed}, U={U_fixed})')
ax.axvline(L_fixed, color='grey', ls=':', lw=1, label='Lower barrier L')
ax.axvline(U_fixed, color='grey', ls='--', lw=1, label='Upper barrier U')
ax.set_xlabel('Strike K')
ax.set_ylabel('Call price')
ax.set_title('Double knock-out call across strikes')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### 1.4  Series convergence in `n_terms`

In [ ]:
N_grid = [1, 2, 3, 5, 10, 20, 50, 100]
conv = [bsm_double_barrier_price(S0, K, L, U, r, q, T, sigma, cp=1, n_terms=n) for n in N_grid]

fig, ax = plt.subplots(figsize=(7, 3))
ax.semilogx(N_grid, conv, 'o-')
ax.axhline(conv[-1], color='r', ls='--', lw=1, label=f'n_terms=100: {conv[-1]:.6f}')
ax.set_xlabel('Number of eigenfunction terms')
ax.set_ylabel('DKO call price')
ax.set_title('Eigenfunction series convergence')
ax.legend()
plt.tight_layout()
plt.show()
print('Converges in fewer than 10 terms for typical parameters.')

---
## Part 2 — SABR Stochastic Volatility

The SABR model (Hagan et al. 2002):

$$dF = \alpha F^\beta \, dW_1, \quad d\alpha = \nu \alpha \, dW_2, \quad \langle dW_1, dW_2 \rangle = \rho \, dt$$

Parameters:
- $\alpha$ — initial volatility level
- $\beta \in [0, 1]$ — CEV backbone (0 = normal, 1 = lognormal)
- $\rho$ — skew correlation
- $\nu$ — vol-of-vol (smile curvature)

### 2.1  ATM behaviour across $\beta$

In [ ]:
F0 = 100.0
betas = np.linspace(0, 1, 11)
alpha = 0.3
atm_vols = [
    float(sabr_hagan_implied_vol(F0, [F0], T, SabrParams(alpha=alpha, beta=b, rho=0.0, nu=0.0))[0])
    for b in betas
]

# Theoretical: sigma_ATM ≈ alpha / F^(1-beta) for nu=0, rho=0 (leading order)
theoretical = [alpha / F0 ** (1 - b) for b in betas]

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(betas, atm_vols, 'bo-', label='Hagan ATM vol')
ax.plot(betas, theoretical, 'r--', label=r'$\alpha / F^{1-\beta}$ (leading order)')
ax.set_xlabel('β')
ax.set_ylabel('ATM implied vol')
ax.set_title(f'SABR ATM vol vs β  (α={alpha}, ν=0, ρ=0, F₀={F0})')
ax.legend()
plt.tight_layout()
plt.show()

### 2.2  Vol smile: effect of $\nu$ (vol-of-vol)

In [ ]:
strikes_smile = np.linspace(70, 140, 200)
nu_values = [0.0, 0.2, 0.4, 0.8]

fig, ax = plt.subplots(figsize=(9, 4))
for nu_val in nu_values:
    p = SabrParams(alpha=0.3, beta=0.5, rho=0.0, nu=nu_val)
    vols = sabr_hagan_implied_vol(F0, strikes_smile, T, p)
    ax.plot(strikes_smile, vols * 100, label=f'ν = {nu_val}')

ax.axvline(F0, color='grey', ls=':', lw=1)
ax.set_xlabel('Strike K')
ax.set_ylabel('Implied vol (%)')
ax.set_title('SABR smile — effect of ν  (α=0.30, β=0.5, ρ=0)')
ax.legend()
plt.tight_layout()
plt.show()

### 2.3  Vol smile: effect of $\rho$ (skew)

In [ ]:
rho_values = [-0.7, -0.3, 0.0, 0.3, 0.7]

fig, ax = plt.subplots(figsize=(9, 4))
for rho_val in rho_values:
    p = SabrParams(alpha=0.3, beta=0.5, rho=rho_val, nu=0.5)
    vols = sabr_hagan_implied_vol(F0, strikes_smile, T, p)
    ax.plot(strikes_smile, vols * 100, label=f'ρ = {rho_val:+.1f}')

ax.axvline(F0, color='grey', ls=':', lw=1)
ax.set_xlabel('Strike K')
ax.set_ylabel('Implied vol (%)')
ax.set_title('SABR smile — effect of ρ  (α=0.30, β=0.5, ν=0.5)')
ax.legend()
plt.tight_layout()
plt.show()

### 2.4  Effect of $\beta$: backbone interpolation

In [ ]:
beta_values = [0.0, 0.25, 0.5, 0.75, 1.0]

fig, ax = plt.subplots(figsize=(9, 4))
for b in beta_values:
    # Set alpha so ATM vol ≈ 20% for each beta
    alpha_atm = 0.20 * F0 ** (1 - b)
    p = SabrParams(alpha=alpha_atm, beta=b, rho=-0.3, nu=0.5)
    vols = sabr_hagan_implied_vol(F0, strikes_smile, T, p)
    ax.plot(strikes_smile, vols * 100, label=f'β = {b}')

ax.axvline(F0, color='grey', ls=':', lw=1)
ax.set_xlabel('Strike K')
ax.set_ylabel('Implied vol (%)')
ax.set_title('SABR smile — effect of β  (ρ=−0.3, ν=0.5, ATM vol ≈ 20%)')
ax.legend()
plt.tight_layout()
plt.show()

### 2.5  Option price strip via `price_strip`

In [ ]:
fwd = ForwardSpec(S0=F0, r=0.05, q=0.0, T=1.0)
p_sabr = SabrParams(alpha=0.30, beta=0.5, rho=-0.3, nu=0.4)

strikes_strip = np.array([80, 85, 90, 95, 100, 105, 110, 115, 120], dtype=float)
sabr_calls = price_strip('sabr', 'sabr_hagan', strikes_strip, fwd, p_sabr, cp=1)

# Flat-vol BSM at ATM vol for comparison
from foureng.models.bsm import BsmParams
from foureng.analytics.bsm_barrier import bsm_call as _bsm_call
atm_vol = float(sabr_hagan_implied_vol(F0, [F0], T, p_sabr)[0])
flat_bsm_calls = np.array([_bsm_call(F0, k, 0.05, 0.0, T, atm_vol) for k in strikes_strip])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: prices
axes[0].plot(strikes_strip, sabr_calls, 'bo-', label='SABR (Hagan)')
axes[0].plot(strikes_strip, flat_bsm_calls, 'r--', label=f'Flat-vol BSM (σ={atm_vol:.2%})')
axes[0].set_xlabel('Strike')
axes[0].set_ylabel('Call price')
axes[0].set_title('SABR vs flat-vol call prices')
axes[0].legend()

# Right: implied vols
sabr_vols = sabr_hagan_implied_vol(F0, strikes_strip, T, p_sabr)
axes[1].plot(strikes_strip, sabr_vols * 100, 'bo-', label='SABR implied vol')
axes[1].axhline(atm_vol * 100, color='r', ls='--', label=f'ATM vol = {atm_vol:.2%}')
axes[1].set_xlabel('Strike')
axes[1].set_ylabel('Implied vol (%)')
axes[1].set_title('SABR implied vol smile')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'SABR params: α={p_sabr.alpha}, β={p_sabr.beta}, ρ={p_sabr.rho}, ν={p_sabr.nu}')
print(f'ATM implied vol: {atm_vol:.4f}')

### 2.6  Put-call parity check

In [ ]:
sabr_puts = price_strip('sabr', 'sabr_hagan', strikes_strip, fwd, p_sabr, cp=-1)

# PCP: C - P = (F0 - K) * e^{-rT} where F0 = S0 * e^{(r-q)T}
F0_fwd = fwd.S0 * np.exp((fwd.r - fwd.q) * fwd.T)
pcp_theoretical = (F0_fwd - strikes_strip) * np.exp(-fwd.r * fwd.T)
pcp_actual = sabr_calls - sabr_puts

print('Put-call parity check (C − P vs (F₀−K)e^{−rT})')
print(f'{"Strike":>8}  {"C−P":>10}  {"Theoretical":>12}  {"Error":>10}')
for k, cp_act, cp_th in zip(strikes_strip, pcp_actual, pcp_theoretical):
    print(f'{k:8.0f}  {cp_act:10.4f}  {cp_th:12.4f}  {abs(cp_act-cp_th):10.2e}')